# 📧 Email Marketing — Send Planning

**Read-only send-planning views: active campaigns, email sequences, lead-campaign assignments, and send eligibility.**

| Section | Purpose | Databases |
|---------|---------|-----------|
| 1 — Campaigns & Templates | `read_campaigns()`, `read_templates()`, `read_leads()` | campaigns, email_sequences, leads_crm |
| 2 — Lead-Campaign View | Which leads are assigned to which campaigns | leads_crm, campaigns |
| 3 — Send Planning | Next emails eligible to send per campaign | campaigns, email_sequences, leads_crm |

> **Note:** Campaign management (create/activate/pause), template creation, and lead assignment are in [Campaign_Manager.ipynb](Campaign_Manager.ipynb). This notebook is **read-only**.

In [1]:
# ── Cell 1: Setup & Configuration ────────────────────────────────────
import sys, os

_WS_ROOT = r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"
_MARKETING_DIR = os.path.join(_WS_ROOT, "Business", "GROWTH", "executions", "Marketing")

# Ensure both workspace root and marketing package parent are importable
for p in [_WS_ROOT, _MARKETING_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(os.path.join(_WS_ROOT, ".env"))

from shared.notion_client import DATABASES
import shared.notion_client as _nc

# ── Notebook-safe patch: add relation handling once only ──
# Re-running this cell must not wrap _extract_value again, otherwise
# the wrapper chain can recurse back into itself.
if not getattr(_nc, "_send_emails_relation_patch_applied", False):
    _nc._send_emails_base_extract = _nc._extract_value

    def _patched_extract(prop):
        if prop.get("type") == "relation":
            return [r["id"] for r in prop.get("relation", [])]
        return _nc._send_emails_base_extract(prop)

    _nc._extract_value = _patched_extract
    _nc._send_emails_relation_patch_applied = True
    print("  Applied relation patch for notebook reads")
else:
    print("  Relation patch already active")

# ── Import read-only modules only ──
from email_marketing.campaigns import read_campaigns
from email_marketing.templates import read_templates, list_templates_for_campaign
from email_marketing.leads import read_leads, _read_leads_full
from email_marketing.config import SEGMENTS

print("Setup complete  (read-only send-planning mode)")
print(f"  NOTION_API_KEY set: {bool(os.getenv('NOTION_API_KEY') or os.getenv('NOTION_TOKEN'))}")
print(f"  Campaigns DB:       {DATABASES['campaigns'][:8]}...")
print(f"  Email Sequences DB: {DATABASES['email_sequences'][:8]}...")
print(f"  Leads CRM DB:       {DATABASES['leads_crm'][:8]}...")
print(f"  Read functions:     read_campaigns  read_templates  read_leads  list_templates_for_campaign")

python-dotenv could not parse statement starting at line 129
python-dotenv could not parse statement starting at line 130
python-dotenv could not parse statement starting at line 129
python-dotenv could not parse statement starting at line 130
python-dotenv could not parse statement starting at line 129
python-dotenv could not parse statement starting at line 130
python-dotenv could not parse statement starting at line 129
python-dotenv could not parse statement starting at line 130


  Applied relation patch for notebook reads
Setup complete  (read-only send-planning mode)
  NOTION_API_KEY set: True
  Campaigns DB:       38566e9d...
  Email Sequences DB: 2258cf16...
  Leads CRM DB:       30b652ea...
  Read functions:     read_campaigns  read_templates  read_leads  list_templates_for_campaign


## Section 1 — Notion Read

Read-only queries against all three databases. Returns clean Python dicts with both writable and read-only fields.

In [2]:
# ── Section 1: Read Functions ─────────────────────────────────────────
# Imported from execution/campaigns.py, execution/templates.py, execution/leads.py
#   read_campaigns(status_filter=None)
#   read_templates(campaign_id=None)
#   read_leads(segment=None, exclude_unsubscribed=True)

print("Read functions imported: read_campaigns(), read_templates(), read_leads()")

Read functions imported: read_campaigns(), read_templates(), read_leads()


In [3]:
# ── Section 1 Demo: Read all three databases ─────────────────────────

from shared.notion_client import notion_request, _API_BASE

# Campaigns
campaigns = read_campaigns()
print(f"=== CAMPAIGNS ({len(campaigns)}) ===")
for c in campaigns:
    metrics = f"score={c['campaign_score']}  open={c['open_rate']}  click={c['click_rate']}  bounce={c['bounce_rate']}  reply={c['reply_rate']}"
    print(f"  [{c['status']:>15}] {c['name']:<35} freq={c['send_frequency']}h  {metrics}")

# Templates (all)
templates = read_templates()
print(f"\n=== EMAIL TEMPLATES ({len(templates)}) ===")
for t in templates:
    metrics = f"open={t['open_rate']}  click={t['click_rate']}  bounce={t['bounce_rate']}  reply={t['reply_rate']}"
    print(f"  {t['template_name']:<40} subject: {t['subject_line'][:40]}  {metrics}")

# Leads preview (first page only — avoids slow full-table scan)
print("\n=== LEADS PREVIEW (first 10 rows) ===")
resp = notion_request(
    "post",
    f"{_API_BASE}/databases/{DATABASES['leads_crm']}/query",
    headers={
        "Authorization": f"Bearer {os.getenv('NOTION_API_KEY') or os.getenv('NOTION_TOKEN')}",
        "Notion-Version": "2022-06-28",
        "Content-Type": "application/json",
    },
    json={"page_size": 10},
    timeout=60,
)
resp.raise_for_status()
data = resp.json()
for page in data.get("results", []):
    props = page.get("properties", {})
    email = props.get("Email", {}).get("email") or ""
    first_name = "".join(item.get("plain_text", "") for item in props.get("First Name", {}).get("rich_text", []))
    source_prop = props.get("Source", {})
    source = (source_prop.get("select") or {}).get("name", "")
    unsubscribed = props.get("Unsubscribed", {}).get("checkbox", False)
    print(f"  {email:<35} first_name={first_name or '-':<12} unsub={unsubscribed}  src={source}")

print(f"\nPreviewed {len(data.get('results', []))} leads from leads_crm (first page only)")

=== CAMPAIGNS (6) ===
  [         Active] Hedge Awareness Campaign            freq=36h  score=50  open=26  click=4  bounce=1  reply=0
  [         Active] Re-Engagement Campaign              freq=72h  score=80  open=0  click=0  bounce=0  reply=0
  [In building phase] The Hedge Report Newsletter         freq=336h  score=10  open=0  click=0  bounce=0  reply=0
  [         Active] Warm Lead Conversion                freq=72h  score=70  open=50  click=9  bounce=0  reply=0
  [         Active] hot-lead follow up                  freq=48h  score=90  open=89  click=35  bounce=0  reply=0
  [         Active] Beta Activation                     freq=48h  score=90  open=0  click=0  bounce=0  reply=0

=== EMAIL TEMPLATES (18) ===
  Email 7                                  subject: $19/mo vs $500 per failed challenge  open=24  click=6  bounce=0  reply=0
  Email 6                                  subject: Six challenges. $1,100. Then one thing c  open=18  click=4  bounce=1  reply=0
  Email 1           

## Section 2 — Lead-Campaign View

Which leads are currently assigned to each active campaign. Read-only cross-reference of leads_crm and campaigns databases.

In [4]:
# ── Section 4 Demo: Which leads are in which campaigns? ───────────────
# Full view: load all leads with campaign relations, cross-reference campaign names

leads_all, email_to_rel = _read_leads_full(exclude_unsubscribed=True)
campaigns_all = read_campaigns()
camp_id_to_name = {c["id"]: c["name"] for c in campaigns_all}

# Build per-campaign lead lists
campaign_leads = {}
unassigned = []
for lead in leads_all:
    rel_ids = email_to_rel.get(lead["email"], [])
    if not rel_ids:
        unassigned.append(lead)
    else:
        for cid in rel_ids:
            cname = camp_id_to_name.get(cid, cid[:8] + "...")
            campaign_leads.setdefault(cname, []).append(lead)

print("=" * 70)
print("  LEAD → CAMPAIGN ASSIGNMENTS")
print("=" * 70)
for cname, members in sorted(campaign_leads.items(), key=lambda x: -len(x[1])):
    print(f"\n  📧 {cname}  ({len(members)} leads)")
    for m in members[:5]:
        print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
    if len(members) > 5:
        print(f"      ... and {len(members) - 5} more")

print(f"\n  ⚠️  Unassigned leads: {len(unassigned)}")
for m in unassigned[:5]:
    print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
if len(unassigned) > 5:
    print(f"      ... and {len(unassigned) - 5} more")

# Summary
total_assigned = sum(len(v) for v in campaign_leads.values())
print(f"\n{'─' * 70}")
print(f"  Total leads: {len(leads_all)} | Assigned: {total_assigned} | Unassigned: {len(unassigned)}")

  LEAD → CAMPAIGN ASSIGNMENTS

  📧 Hedge Awareness Campaign  (68 leads)
      leonkazungu5@gmail.com              [Cold] score=2
      adeshthosani76@gmail.com            [Cold] score=2
      muhammed.olafusi@gmail.com          [Cold] score=2
      diegogaliachoo@gmail.com            [Cold] score=1
      vasu9167@gmail.com                  [Cold] score=1
      ... and 63 more

  📧 Warm Lead Conversion  (64 leads)
      rickhjlee@gmail.com                 [Warm] score=7
      ryanghasry84@gmail.com              [Warm] score=6
      hoangvanhieu.hvnh@gmail.com         [Warm] score=7
      christianmoodie2@gmail.com          [Warm] score=3
      drnahuelfinance@gmail.com           [Warm] score=6
      ... and 59 more

  📧 hot-lead follow up  (5 leads)
      aldenlee@synergy.com.sg             [Hot] score=27
      oasishalifax@hotmail.com            [Hot] score=23
      mizraubaid1190@gmail.com            [Hot] score=24
      mdsahil1120@gmail.com               [Hot] score=25
      jamiefx

In [5]:
# ── Section 5: What emails need sending next? ─────────────────────────
# For each active campaign, show the next template in the sequence
# and which leads are eligible to receive it.

from datetime import datetime, timezone, timedelta

active_camps = read_campaigns(status_filter="Active")
leads_all2, email_to_rel2 = _read_leads_full(exclude_unsubscribed=True)

def _parse_last_send(val):
    """Extract an ISO string from last_send (may be str, dict, or None)."""
    if not val:
        return ""
    if isinstance(val, dict):
        return val.get("start") or ""
    return str(val)

print("=" * 70)
print("  NEXT EMAILS TO SEND")
print("=" * 70)

total_pending = 0
for camp in sorted(active_camps, key=lambda c: -c["campaign_score"]):
    cid = camp["id"]
    freq_hours = camp["send_frequency"]
    tpls = list_templates_for_campaign(cid)
    
    # Find leads assigned to this campaign
    assigned = [l for l in leads_all2 if cid in email_to_rel2.get(l["email"], [])]
    
    if not assigned:
        print(f"\n  {camp['name']}  (score={camp['campaign_score']}, freq={freq_hours}h)")
        print(f"      No leads assigned - skipping")
        continue
    
    # Check send eligibility: leads whose last_send was > freq_hours ago (or never)
    now = datetime.now(timezone.utc)
    eligible = []
    for lead in assigned:
        last = _parse_last_send(lead.get("last_send"))
        if not last:
            eligible.append(lead)
            continue
        try:
            last_dt = datetime.fromisoformat(last.replace("Z", "+00:00"))
            if (now - last_dt) >= timedelta(hours=freq_hours):
                eligible.append(lead)
        except (ValueError, TypeError):
            eligible.append(lead)
    
    print(f"\n  {camp['name']}  (score={camp['campaign_score']}, freq={freq_hours}h)")
    print(f"      Templates: {len(tpls)} | Assigned: {len(assigned)} | Eligible now: {len(eligible)}")
    
    if tpls:
        print(f"      Sequence:")
        for t in tpls:
            print(f"        #{t['_seq_num']} {t['template_name']:<30} -> \"{t['subject_line'][:45]}\"")
    
    if eligible:
        print(f"      Eligible leads (first 5):")
        for e in eligible[:5]:
            ls = _parse_last_send(e['last_send'])
            last_str = ls[:10] if ls else 'never'
            print(f"        {e['email']:<35} [{e['segment']}] score={e['score']}  last_send={last_str}")
        if len(eligible) > 5:
            print(f"        ... and {len(eligible) - 5} more")
    
    total_pending += len(eligible)

print(f"\n{'=' * 70}")
print(f"  TOTAL EMAILS PENDING: {total_pending} leads eligible across {len(active_camps)} active campaigns")

  NEXT EMAILS TO SEND

  hot-lead follow up  (score=90, freq=48h)
      Templates: 3 | Assigned: 5 | Eligible now: 5
      Sequence:
        #1 Hot lead email 2               -> "Where most traders lose it (not the trading)"
        #2 Hot lead email 1               -> "Quick one — {{name}}, exact setup mapped out"
        #3 Hot lead email 3               -> "Should I close your file for now?"
      Eligible leads (first 5):
        aldenlee@synergy.com.sg             [Hot] score=27  last_send=2026-03-09
        oasishalifax@hotmail.com            [Hot] score=23  last_send=2026-03-18
        mizraubaid1190@gmail.com            [Hot] score=24  last_send=2026-03-12
        mdsahil1120@gmail.com               [Hot] score=25  last_send=2026-03-12
        jamiefxinvesting@gmail.com          [Hot] score=36  last_send=2026-03-12

  Beta Activation  (score=90, freq=48h)
      Templates: 3 | Assigned: 3 | Eligible now: 0
      Sequence:
        #1 Beta 1                         -> "You're in. 